# 🚀 DT-RL: End-to-End Training Pipeline

**Self-contained notebook** that trains the full Dependency-Aware Trading Pipeline on **real Dow 30 data**.

| Module | Purpose |
|--------|---------|
| Module 1 | Stock data alignment & validity masks |
| Module 2 | Hypergraph cross-asset encoder |
| Module 3 | Continuous regime classifier (4 regimes) |
| Module 4 | Decision Transformer (offline RL) |
| Module 5 | Portfolio weights / Forecasting output |

---

> **Prerequisites** (Colab Secrets — key icon on the left sidebar):
> - `GITHUB_TOKEN`: A GitHub Personal Access Token with repo scope.
> - `WANDB_API_KEY`: Your Weights & Biases API Key (optional).

## 1. ⚙️ Setup & Repository Sync

In [ ]:
import os
import subprocess
import sys

# ==========================================
# CONFIGURATION
# ==========================================
GITHUB_USER = "VvS-2403"                   # Your GitHub username
GITHUB_REPO = "DT-RL"                      # The repository name
GITHUB_EMAIL = "your-email@example.com"    # Email for git commits
GITHUB_NAME = "Automated Colab Runner"     # Name for git commits
BRANCH = "main"                            # Branch to pull from / push to
# ==========================================

# 1. Fetch GitHub Token from Colab Secrets
try:
    from google.colab import userdata
    GIT_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ GitHub Token found in Colab Secrets.")
    IN_COLAB = True
except Exception:
    print("⚠️  Not running in Colab or GITHUB_TOKEN not set. Assuming local run.")
    GIT_TOKEN = None
    IN_COLAB = False

# 2. Clone or Update the Repository
if GIT_TOKEN:
    REPO_URL = f"https://{GIT_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    if not os.path.exists(GITHUB_REPO):
        print(f"📥 Cloning repository: {GITHUB_REPO}...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    else:
        print(f"🔄 Repository {GITHUB_REPO} exists. Pulling latest changes...")
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=GITHUB_REPO, check=True)
    os.chdir(GITHUB_REPO)

# 3. Ensure project root is on sys.path
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"📁 Working directory: {os.getcwd()}")

## 2. 📦 Install Dependencies & Imports

In [ ]:
# Install requirements (only needed on Colab)
if IN_COLAB:
    print("📦 Installing dependencies...")
    !pip install -q torch numpy pandas matplotlib seaborn wandb tqdm scipy pyarrow pyyaml
    print("✅ Dependencies installed.")
else:
    print("ℹ️  Assuming local environment has dependencies installed.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import math
import warnings
from pathlib import Path
from datetime import datetime
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Plot styling
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11})

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if device == "cuda":
    torch.cuda.manual_seed_all(42)

## 3. 🔑 Weights & Biases Authentication (Optional)

In [ ]:
USE_WANDB = True  # Set to False to skip WandB logging entirely

if USE_WANDB:
    try:
        import wandb
        if IN_COLAB:
            WANDB_KEY = userdata.get('WANDB_API_KEY')
            os.environ["WANDB_API_KEY"] = WANDB_KEY
            wandb.login(key=WANDB_KEY)
        else:
            wandb.login()
        print("✅ WandB authenticated")
    except Exception as e:
        print(f"⚠️  WandB login failed: {e}. Continuing without WandB.")
        USE_WANDB = False

## 4. 📊 Configuration

All hyperparameters in one place. These defaults are **optimized for the Dow 30 real data**.

In [ ]:
# ═══════════════════════════════════════════
# CONFIGURATION — Edit these values as needed
# ═══════════════════════════════════════════

CONFIG = {
    # Data
    "lookback_window": 60,        # Lookback window (trading days)
    "forecast_horizon": 5,        # Forecast horizon (trading days)
    "stride": 5,                  # Stride between training sequences
    "seed": 42,

    # Model architecture
    "output_mode": "portfolio_weights",   # "portfolio_weights" or "forecasting"
    "num_regimes": 4,
    "embedding_dim": 128,
    "num_transformer_layers": 4,
    "num_attention_heads": 8,

    # Training — optimized for transformer convergence on real data
    "batch_size": 32,             # Good GPU utilization on T4/A100
    "learning_rate": 1e-4,        # Lower LR critical for transformer stability
    "weight_decay": 1e-5,         # Regularization
    "num_epochs": 100,            # Enough for convergence with early stopping
    "early_stopping_patience": 15,# Patient enough to get through LR plateaus
    "warmup_epochs": 5,           # Linear warmup prevents early divergence
    "gradient_clip": 1.0,         # Gradient clipping for stability
    "gradient_accumulation_steps": 2,  # Effective batch = 64

    # Data split
    "train_split": 0.8,           # 80% train, 20% validation

    # WandB
    "wandb_project": "DT-RL",
}

print("📋 Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

## 5. 📈 Load Real Dow 30 Market Data

In [ ]:
from ml.data.synthetic_data import SyntheticDataLoader
from ml.modules.module1_input import MarketTensor, StockDataProcessor, DataValidator

# ── Load the real CSV ──────────────────────────────────────────────────
CSV_PATH = "dow30_market_data_2013_2022.csv"

if not os.path.exists(CSV_PATH):
    print(f"⚠️  {CSV_PATH} not found. Attempting to generate with download_dow30.py...")
    try:
        subprocess.run([sys.executable, "download_dow30.py"], check=True)
    except Exception as e:
        raise FileNotFoundError(
            f"Could not find or generate {CSV_PATH}. "
            f"Please upload the CSV file manually or run download_dow30.py first.\n"
            f"Error: {e}"
        )

print(f"📂 Loading market data from {CSV_PATH}...")
raw_df = pd.read_csv(CSV_PATH)
raw_df['Date'] = pd.to_datetime(raw_df['Date'])
raw_df = raw_df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

num_assets = raw_df['Ticker'].nunique()
num_days = raw_df['Date'].nunique()
tickers = sorted(raw_df['Ticker'].unique())

print(f"\n📊 Dataset Summary:")
print(f"   Rows: {len(raw_df):,}")
print(f"   Assets (N): {num_assets}")
print(f"   Trading Days (T): {num_days}")
print(f"   Date range: {raw_df['Date'].min().date()} → {raw_df['Date'].max().date()}")
print(f"   Columns: {list(raw_df.columns)}")
print(f"   Tickers: {tickers}")

In [ ]:
# ── Process into MarketTensor ──────────────────────────────────────────
processor = StockDataProcessor(lookback_window=CONFIG["lookback_window"])
market_tensor = processor.process(raw_df)

# Validate
validator = DataValidator()
report = validator.validate_market_tensor(market_tensor)

print(f"\n📋 Data Validation:")
print(f"   Valid: {'✅' if report['valid'] else '❌'}")
print(f"   Shape: N={report['shape']['num_assets']}, "
      f"T={report['shape']['lookback_window']}, "
      f"F={report['shape']['num_features']}")
print(f"   Coverage: {report['coverage']['overall']:.1%}")
print(f"   Features: {market_tensor.feature_names}")
if not report["valid"]:
    for issue in report["issues"]:
        print(f"   ⚠️  {issue}")

In [ ]:
# ── Create training sequences ─────────────────────────────────────────
loader = SyntheticDataLoader()
X, M, Y = loader.create_sequences(
    market_tensor,
    lookback=CONFIG["lookback_window"],
    horizon=CONFIG["forecast_horizon"],
    stride=CONFIG["stride"],
)

print(f"\n🧮 Training Sequences:")
print(f"   X (features):  {X.shape}  →  (sequences, assets, lookback, features)")
print(f"   M (mask):      {M.shape}  →  (sequences, assets, lookback)")
print(f"   Y (targets):   {Y.shape}  →  (sequences, assets, horizon)")
print(f"   Num sequences: {X.shape[0]}")
print(f"   Target stats:  mean={Y.mean():.6f}, std={Y.std():.6f}, min={Y.min():.4f}, max={Y.max():.4f}")

## 6. 🔍 Data Visualizations

In [ ]:
from ml.visualization import plot_price_trajectories, plot_coverage_heatmap

# Price trajectories
prices = market_tensor.features[:, :, 3].numpy()  # Close prices (normalized)
fig_prices = plot_price_trajectories(
    prices, market_tensor.asset_ids, max_assets=10,
    title="Dow 30 Price Trajectories (z-score normalized)",
)
plt.show()
plt.close(fig_prices)

In [ ]:
# Coverage heatmap
fig_coverage = plot_coverage_heatmap(
    market_tensor.validity_mask.numpy(),
    market_tensor.asset_ids,
    title="Dow 30 Data Coverage (2013-2022)",
)
plt.show()
plt.close(fig_coverage)

In [ ]:
# Feature correlation matrix (first asset)
features = market_tensor.features.numpy()
fig, ax = plt.subplots(figsize=(8, 6))
feature_data = features[0, :, :].T
corr = np.corrcoef(feature_data)
feature_names = market_tensor.feature_names[:len(corr)]
sns.heatmap(corr, xticklabels=feature_names, yticklabels=feature_names,
            cmap="coolwarm", center=0, annot=True, fmt=".2f",
            ax=ax, square=True, linewidths=0.5)
ax.set_title(f"Feature Correlation Matrix ({market_tensor.asset_ids[0]})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
plt.close(fig)

## 7. 🧠 Pipeline Initialization

In [ ]:
from ml.pipeline import DependencyAwareTradingPipeline, PipelineConfig

config = PipelineConfig(
    lookback_window=CONFIG["lookback_window"],
    forecast_horizon=CONFIG["forecast_horizon"],
    output_mode=CONFIG["output_mode"],
    batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    num_epochs=CONFIG["num_epochs"],
    early_stopping_patience=CONFIG["early_stopping_patience"],
    gradient_clip=CONFIG["gradient_clip"],
    num_transformer_layers=CONFIG["num_transformer_layers"],
    num_attention_heads=CONFIG["num_attention_heads"],
    device=device,
)

pipeline = DependencyAwareTradingPipeline(config).to(device)

# Initialize modules with data shape
dummy_mt = MarketTensor(
    features=torch.zeros(X.shape[1], X.shape[2], X.shape[3]),
    validity_mask=torch.ones(X.shape[1], X.shape[2]),
    asset_ids=market_tensor.asset_ids,
    timestamps=np.arange(X.shape[2]),
    feature_names=market_tensor.feature_names,
    feature_means=torch.zeros(X.shape[3]),
    feature_stds=torch.ones(X.shape[3]),
)
pipeline.initialize_modules(dummy_mt)

# Model summary
print(f"\n🧠 Model Architecture:")
total_params = 0
param_counts = {}
for name, module in [("Module2_Encoder", pipeline.module2),
                     ("Module3_Regime", pipeline.module3),
                     ("Module4_Transformer", pipeline.module4),
                     ("Module5_Output", pipeline.module5)]:
    count = sum(p.numel() for p in module.parameters())
    param_counts[name] = count
    total_params += count
    print(f"   {name}: {count:,} parameters")
print(f"   {'─'*40}")
print(f"   Total: {total_params:,} parameters")

In [ ]:
from ml.visualization import plot_model_summary

fig_params = plot_model_summary(param_counts)
plt.show()
plt.close(fig_params)

## 8. 🏋️ Training

Full inline training with:
- **80/20 chronological train/val split** (no future leakage)
- **Linear LR warmup** → **Cosine annealing** schedule
- **Gradient accumulation** (effective batch = batch_size × accum_steps)
- **Mixed precision** on CUDA for speed
- **Early stopping** with patience

In [ ]:
# ── Chronological 80/20 train/val split ────────────────────────────────
n_total = X.shape[0]
n_train = int(n_total * CONFIG["train_split"])
n_val = n_total - n_train

X_train, M_train, Y_train = X[:n_train], M[:n_train], Y[:n_train]
X_val, M_val, Y_val = X[n_train:], M[n_train:], Y[n_train:]

train_loader = DataLoader(TensorDataset(X_train, M_train, Y_train),
                          batch_size=CONFIG["batch_size"], shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, M_val, Y_val),
                        batch_size=CONFIG["batch_size"], shuffle=False)

print(f"📊 Data Split (80/20 chronological — no future leakage):")
print(f"   Train: {n_train} sequences ({CONFIG['train_split']:.0%})")
print(f"   Val:   {n_val} sequences ({1 - CONFIG['train_split']:.0%})")

In [ ]:
# ── Initialize WandB run ───────────────────────────────────────────────
if USE_WANDB:
    import wandb
    run = wandb.init(
        project=CONFIG["wandb_project"],
        name=f"e2e_{CONFIG['output_mode']}_{datetime.now().strftime('%m%d_%H%M%S')}",
        config=CONFIG,
        reinit=True,
    )
    wandb.log({"model/total_parameters": total_params})
    print(f"📊 WandB Dashboard: {run.url}")

In [ ]:
from ml.training import SequenceTrainer

# ── Setup optimizer, scheduler, and trainer ────────────────────────────
trainer = SequenceTrainer(pipeline, config, checkpoint_dir=Path("checkpoints"))

# Replace the default optimizer with AdamW (better weight decay handling)
trainer.optimizer = torch.optim.AdamW(
    pipeline.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.999),
    eps=1e-8,
)

# Warmup + Cosine Annealing LR schedule
warmup_epochs = CONFIG["warmup_epochs"]
total_epochs = CONFIG["num_epochs"]

def lr_lambda(epoch):
    """Linear warmup for warmup_epochs, then cosine decay."""
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(trainer.optimizer, lr_lambda)
# Override the ReduceLROnPlateau scheduler in trainer
trainer.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    trainer.optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6,
)

# Mixed precision setup
use_amp = (device == "cuda")
scaler = torch.amp.GradScaler(device) if use_amp else None

accum_steps = CONFIG["gradient_accumulation_steps"]

print(f"\n{'═'*70}")
print(f"🏋️  Training for {total_epochs} epochs (80/20 train/val split)")
print(f"   Effective batch size: {CONFIG['batch_size']} × {accum_steps} = {CONFIG['batch_size'] * accum_steps}")
print(f"   LR schedule: warmup({warmup_epochs} epochs) → cosine + ReduceLROnPlateau")
print(f"   Mixed precision: {'ON (AMP)' if use_amp else 'OFF (CPU)'}")
print(f"   Gradient clipping: {CONFIG['gradient_clip']}")
print(f"{'═'*70}\n")

# ── Training loop ──────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "lr": []}
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(total_epochs):
    t0 = time.time()

    # ── Train epoch ────────────────────────────────────────────────
    pipeline.train()
    total_train_loss = 0.0
    num_train_batches = 0
    trainer.optimizer.zero_grad()

    for batch_idx, (X_b, M_b, Y_b) in enumerate(train_loader):
        X_b = X_b.to(device)
        M_b = M_b.to(device)
        Y_b = Y_b.to(device)

        # Construct return-to-go
        rtg = trainer._construct_return_to_go(Y_b, X_b.shape[2])

        if use_amp:
            with torch.amp.autocast(device):
                output = pipeline(X_b, M_b, rtg, actions=None)
                loss = trainer._compute_loss(output, Y_b)
                loss = loss / accum_steps  # Scale for accumulation

            scaler.scale(loss).backward()

            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
                scaler.unscale_(trainer.optimizer)
                torch.nn.utils.clip_grad_norm_(pipeline.parameters(), CONFIG["gradient_clip"])
                scaler.step(trainer.optimizer)
                scaler.update()
                trainer.optimizer.zero_grad()
        else:
            output = pipeline(X_b, M_b, rtg, actions=None)
            loss = trainer._compute_loss(output, Y_b)
            loss = loss / accum_steps
            loss.backward()

            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(pipeline.parameters(), CONFIG["gradient_clip"])
                trainer.optimizer.step()
                trainer.optimizer.zero_grad()

        total_train_loss += loss.item() * accum_steps  # Unscale for logging
        num_train_batches += 1

    train_loss = total_train_loss / num_train_batches

    # ── Validate ───────────────────────────────────────────────────
    val_loss = trainer.validate(val_loader)

    current_lr = trainer.optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    # Print progress
    marker = " ⭐" if val_loss < best_val_loss else ""
    print(f"  Epoch {epoch+1:3d}/{total_epochs} | "
          f"Train={train_loss:.6f} | Val={val_loss:.6f} | "
          f"LR={current_lr:.2e} | {elapsed:.1f}s{marker}")

    # ── WandB logging ──────────────────────────────────────────────
    if USE_WANDB:
        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/learning_rate": current_lr,
            "train/epoch_time": elapsed,
        }, step=epoch + 1)

    # ── LR schedulers ──────────────────────────────────────────────
    scheduler.step()  # Cosine/warmup
    trainer.scheduler.step(val_loss)  # ReduceLROnPlateau

    # ── Early stopping + checkpointing ─────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        trainer._save_checkpoint(epoch, val_loss)
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"\n⏹️  Early stopping at epoch {epoch+1} (patience={CONFIG['early_stopping_patience']})")
            break

    # Free GPU memory
    if device == "cuda":
        torch.cuda.empty_cache()

# ── Load best checkpoint ───────────────────────────────────────────────
best_ckpt = sorted(Path("checkpoints").glob("best_*.pt"))
if best_ckpt:
    ckpt = torch.load(best_ckpt[-1], map_location=device, weights_only=False)
    pipeline.load_state_dict(ckpt["state_dict"])
    print(f"\n✅ Loaded best checkpoint: {best_ckpt[-1].name} (val_loss={ckpt['val_loss']:.6f})")

print(f"\n{'═'*70}")
print(f"🎉 Training complete! Best val loss: {best_val_loss:.6f} | Epochs: {len(history['train_loss'])}")
print(f"{'═'*70}")

## 9. 📉 Training Results

In [ ]:
from ml.visualization import plot_loss_curves

fig_loss = plot_loss_curves(history, title="DT-RL Training Loss Curves (Dow 30)")
plt.show()
if USE_WANDB:
    wandb.log({"results/loss_curves": wandb.Image(fig_loss)})
plt.close(fig_loss)

In [ ]:
# Learning rate schedule
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(history["lr"]) + 1), history["lr"],
        color="#7B1FA2", linewidth=2, marker="o", markersize=3)
ax.axvline(x=CONFIG["warmup_epochs"], color="gray", linestyle="--", alpha=0.5,
           label=f"Warmup ends (epoch {CONFIG['warmup_epochs']})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule (Warmup + Cosine + ReduceOnPlateau)",
             fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
plt.close(fig)

## 10. 🎯 Regime Analysis

In [ ]:
# Run inference on a validation sample for analysis
pipeline.eval()
with torch.no_grad():
    sample_X = X_val[:1].to(device)
    sample_M = M_val[:1].to(device)
    sample_Y = Y_val[:1].to(device)

    rtg = sample_Y.sum(dim=-1, keepdim=True).unsqueeze(-1).expand(
        -1, -1, CONFIG["lookback_window"], -1
    )
    outputs = pipeline(sample_X, sample_M, rtg)

regime_probs = outputs["regime_probs"].squeeze(0).cpu().numpy()  # (N, T, 4)
asset_ids = market_tensor.asset_ids

print(f"Regime probabilities shape: {regime_probs.shape}")
print(f"  → {regime_probs.shape[0]} assets × {regime_probs.shape[1]} timesteps × {regime_probs.shape[2]} regimes")

In [ ]:
from ml.visualization import plot_regime_evolution, plot_regime_distribution, plot_regime_transitions

# Regime evolution for first 3 assets
for idx in range(min(3, len(asset_ids))):
    fig = plot_regime_evolution(regime_probs, asset_ids, asset_idx=idx)
    plt.show()
    if USE_WANDB:
        wandb.log({f"results/regime_evolution_{asset_ids[idx]}": wandb.Image(fig)})
    plt.close(fig)

In [ ]:
# Regime distribution
fig_dist = plot_regime_distribution(regime_probs, asset_ids)
plt.show()
if USE_WANDB:
    wandb.log({"results/regime_distribution": wandb.Image(fig_dist)})
plt.close(fig_dist)

In [ ]:
# Regime transitions
from ml.modules.module3_regime import RegimeVisualizer

transitions = RegimeVisualizer.compute_regime_transitions(
    outputs["regime_probs"].squeeze(0),
)
fig_trans = plot_regime_transitions(np.array(transitions["transition_matrix"]))
plt.show()
if USE_WANDB:
    wandb.log({"results/regime_transitions": wandb.Image(fig_trans)})
plt.close(fig_trans)

## 11. 💼 Portfolio & Dependency Analysis

In [ ]:
from ml.visualization import plot_portfolio_weights, plot_equity_curve, plot_dependency_matrix
from ml.modules.module2_encoder import DependencyAnalyzer

# Cross-asset dependency matrix
Z = outputs["encoder_embeddings"].squeeze(0).cpu()  # (N, T, D)
M_single = sample_M.squeeze(0).cpu()
dep_matrix = DependencyAnalyzer.compute_dependency_matrix(Z, M_single)

fig_deps = plot_dependency_matrix(dep_matrix.numpy(), asset_ids,
                                  title="Dow 30 Cross-Asset Dependency Matrix")
plt.show()
if USE_WANDB:
    wandb.log({"results/dependency_matrix": wandb.Image(fig_deps)})
plt.close(fig_deps)

In [ ]:
# Portfolio weights (only for portfolio_weights mode)
if CONFIG["output_mode"] == "portfolio_weights":
    weights = outputs["output"].weights.cpu().numpy()
    N = regime_probs.shape[0]
    T = CONFIG["lookback_window"]
    weights_reshaped = weights.reshape(1, N, T, N)
    last_weights = weights_reshaped[0, :, -1, :]  # (N, N)

    fig_w = plot_portfolio_weights(last_weights, asset_ids,
                                   title="Dow 30 Portfolio Weight Allocation")
    plt.show()
    if USE_WANDB:
        wandb.log({"results/portfolio_weights": wandb.Image(fig_w)})
    plt.close(fig_w)

    # Top allocations
    avg_weights = np.abs(last_weights).mean(axis=0)
    sorted_idx = np.argsort(avg_weights)[::-1]
    print("\n📊 Top 10 Average Absolute Allocations:")
    for rank, idx in enumerate(sorted_idx[:10]):
        name = asset_ids[idx] if idx < len(asset_ids) else f"Asset_{idx}"
        print(f"   {rank+1}. {name}: {avg_weights[idx]:.4f}")

In [ ]:
# Equity curve (simulated from target returns)
avg_returns = Y[:, :, 0].mean(dim=1).numpy()  # Average across assets, first horizon
if len(avg_returns) > 5:
    fig_eq = plot_equity_curve(avg_returns,
                               title="Simulated Portfolio Equity Curve (Dow 30)")
    plt.show()
    if USE_WANDB:
        wandb.log({"results/equity_curve": wandb.Image(fig_eq)})
    plt.close(fig_eq)

## 12. 📋 Evaluation Metrics

Evaluate on the **full validation set** for robust financial metrics.

In [ ]:
from ml.evaluation import evaluate_predictions, compute_financial_metrics
from ml.visualization import plot_metrics_table

# ── Single-sample metrics (for regime/portfolio analysis) ──────────────
eval_metrics = evaluate_predictions(outputs, sample_Y, CONFIG["output_mode"])
eval_metrics["final_train_loss"] = history["train_loss"][-1]
eval_metrics["final_val_loss"] = history["val_loss"][-1]
eval_metrics["best_val_loss"] = min(history["val_loss"])
eval_metrics["total_epochs"] = len(history["train_loss"])

fig_met = plot_metrics_table(eval_metrics, title="DT-RL Performance Metrics (Dow 30)")
plt.show()
if USE_WANDB:
    wandb.log({"results/metrics_table": wandb.Image(fig_met)})
plt.close(fig_met)

print("\n📋 Sample Evaluation Metrics:")
for k, v in eval_metrics.items():
    print(f"   {k}: {v:.6f}")

In [ ]:
# ── Full validation set evaluation ────────────────────────────────────
pipeline.eval()
all_val_losses = []
all_step_returns = []

with torch.no_grad():
    for X_b, M_b, Y_b in val_loader:
        X_b = X_b.to(device)
        M_b = M_b.to(device)
        Y_b = Y_b.to(device)

        rtg = Y_b.sum(dim=-1, keepdim=True).unsqueeze(-1).expand(-1, -1, X_b.shape[2], -1)
        out = pipeline(X_b, M_b, rtg)
        loss = trainer._compute_loss(out, Y_b)
        all_val_losses.append(loss.item())

        # Collect portfolio returns for financial metrics
        if CONFIG["output_mode"] == "portfolio_weights":
            w = out["output"].weights.cpu().numpy()
            bs = Y_b.shape[0]
            N_a = Y_b.shape[1]
            w = w.reshape(bs, N_a, X_b.shape[2], N_a)
            pred_w = w[:, :, -1, :]
            avg_tgt = Y_b.mean(dim=-1).cpu().numpy()
            for b in range(bs):
                port_ret = np.dot(pred_w[b], avg_tgt[b]).mean()
                all_step_returns.append(port_ret)

avg_val_loss = np.mean(all_val_losses)
print(f"\n🔬 Full Validation Set Evaluation:")
print(f"   Average val loss: {avg_val_loss:.6f}")
print(f"   Num val batches: {len(all_val_losses)}")

if all_step_returns:
    returns_arr = np.array(all_step_returns)
    fin_metrics = compute_financial_metrics(returns_arr)
    print(f"\n💰 Financial Metrics (Full Validation Set):")
    for k, v in fin_metrics.items():
        print(f"   {k}: {v:.6f}")

    if USE_WANDB:
        for k, v in fin_metrics.items():
            wandb.summary[f"val/{k}"] = v
        for k, v in eval_metrics.items():
            wandb.summary[f"eval/{k}"] = v

## 13. 📊 Training Summary

In [ ]:
from ml.visualization import plot_training_summary

fig_summary = plot_training_summary(
    history,
    eval_metrics,
    regime_probs,
    avg_returns if len(avg_returns) > 5 else None,
)
plt.show()
if USE_WANDB:
    wandb.log({"results/training_summary": wandb.Image(fig_summary)})
plt.close(fig_summary)

## 14. 💾 Save & Push to GitHub

In [ ]:
# Save final model checkpoint
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = checkpoint_dir / f"dtrl_e2e_{timestamp}.pt"
pipeline.save(checkpoint_path)
print(f"✅ Model saved to {checkpoint_path}")

# Save figures
fig_dir = Path("results") / f"e2e_{timestamp}"
fig_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Results directory: {fig_dir}")

# Upload to WandB
if USE_WANDB:
    artifact = wandb.Artifact(
        name=f"dtrl-e2e-model-{timestamp}",
        type="model",
        description="DT-RL pipeline trained on Dow 30 real data (80/20 split)",
    )
    artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(artifact)
    wandb.finish()
    print("✅ WandB run finished and artifact uploaded.")

In [ ]:
# Push results to GitHub (only if in Colab with token)
if GIT_TOKEN:
    print("⚙️ Configuring git user...")
    subprocess.run(["git", "config", "--global", "user.email", GITHUB_EMAIL], check=True)
    subprocess.run(["git", "config", "--global", "user.name", GITHUB_NAME], check=True)

    print("📝 Adding new files to git tracking...")
    subprocess.run(["git", "add", "checkpoints/"], check=True)
    subprocess.run(["git", "add", "results/"], check=True)

    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)

    if status.stdout.strip():
        print("📦 Committing changes...")
        commit_msg = f"E2E training results - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        subprocess.run(["git", "commit", "-m", commit_msg], check=True)

        print("📤 Pushing to GitHub...")
        subprocess.run(["git", "push", REPO_URL, BRANCH], check=True)
        print("✅ Successfully pushed updates to GitHub.")
    else:
        print("ℹ️ No new changes to commit.")
else:
    print("ℹ️ Skipping git push (no GITHUB_TOKEN).")

In [ ]:
print(f"\n{'═'*60}")
print("🎉 ALL DONE!")
print(f"{'═'*60}")
print(f"""
Summary:
  • Trained on REAL Dow 30 data (2013-2022)
  • Assets: {num_assets} stocks
  • Split: {n_train} train / {n_val} val (80/20 chronological)
  • Trained for {len(history['train_loss'])} epochs
  • Best validation loss: {min(history['val_loss']):.6f}
  • Final train loss: {history['train_loss'][-1]:.6f}
  • Model saved to: {checkpoint_path}
  • Output mode: {CONFIG['output_mode']}
  • Total parameters: {total_params:,}

To load the trained model:
  from ml.pipeline import DependencyAwareTradingPipeline
  pipeline = DependencyAwareTradingPipeline.load("{checkpoint_path}")
""")

---

## 🎉 Workflow Complete

Your model has been trained on real Dow 30 data, tracked in WandB, and the artifacts are safely stored.